# Healthcare Operations EDA & Cleaning Notebook

This notebook validates, cleans, and explores the Healthcare Operations Analytics dataset before Tableau and Streamlit reporting.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

raw_path = '../data/raw/healthcare_operations_raw.csv'
df = pd.read_csv(raw_path)
df.head()


## 1. Dataset Overview


In [ ]:
print(df.shape)
df.info()


## 2. Missing Values Check


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]


## 3. Duplicate Check


In [ ]:
print('Duplicate visit_id:', df['visit_id'].duplicated().sum())
print('Duplicate rows:', df.duplicated().sum())


## 4. Cleaning Steps


In [ ]:
clean = df.copy()
clean['visit_date'] = pd.to_datetime(clean['visit_date'], errors='coerce')
clean['department'] = clean['department'].astype(str).str.strip().str.title()

numeric_cols = ['patient_satisfaction_score', 'estimated_cost_per_visit', 'length_of_stay_days', 'er_wait_time_minutes', 'bed_occupancy_rate', 'capacity_strain_score']
for col in numeric_cols:
    clean[col] = pd.to_numeric(clean[col], errors='coerce')
    clean[col] = clean.groupby('department')[col].transform(lambda s: s.fillna(s.median()))

clean = clean.drop_duplicates(subset=['visit_id'])
clean.isna().sum().sort_values(ascending=False).head(10)


## 5. Department Analysis


In [ ]:
dept = clean.groupby('department').agg(
    patients=('patient_id','nunique'),
    avg_los=('length_of_stay_days','mean'),
    avg_wait=('er_wait_time_minutes','mean'),
    occupancy=('bed_occupancy_rate','mean'),
    readmission_rate=('readmitted_30_days','mean'),
    satisfaction=('patient_satisfaction_score','mean'),
    capacity_strain=('capacity_strain_score','mean')
).sort_values('capacity_strain', ascending=False)
dept


## 6. LOS Analysis


In [ ]:
clean.groupby('department')['length_of_stay_days'].mean().sort_values().plot(kind='barh', title='Average LOS by Department')
plt.xlabel('Average LOS Days')
plt.show()


## 7. Readmission Analysis


In [ ]:
(clean.groupby('department')['readmitted_30_days'].mean().sort_values() * 100).plot(kind='barh', title='Readmission Rate by Department')
plt.xlabel('Readmission Rate (%)')
plt.show()


## 8. ER Wait Time Analysis


In [ ]:
clean.groupby('department')['er_wait_time_minutes'].mean().sort_values().plot(kind='barh', title='Average ER Wait Time by Department')
plt.xlabel('Minutes')
plt.show()


## 9. Capacity Strain Analysis


In [ ]:
clean.groupby(['department','shift'])['capacity_strain_score'].mean().unstack().round(1)


## 10. Monthly Trends


In [ ]:
monthly = clean.groupby('month').agg(
    patients=('patient_id','nunique'),
    avg_los=('length_of_stay_days','mean'),
    avg_wait=('er_wait_time_minutes','mean'),
    occupancy=('bed_occupancy_rate','mean'),
    satisfaction=('patient_satisfaction_score','mean')
).reset_index()
monthly.head()


## 11. Correlation Check


In [ ]:
corr_cols = ['length_of_stay_days','er_wait_time_minutes','throughput_time_hours','bed_occupancy_rate','patient_satisfaction_score','resource_utilization_index','estimated_cost_per_visit','capacity_strain_score']
clean[corr_cols].corr().round(2)


## 12. Save Cleaned Data


In [ ]:
clean.to_csv('../data/cleaned/healthcare_operations_cleaned.csv', index=False)
print('Cleaned dataset saved.')


## Executive Findings

- Review the top departments by capacity strain and wait time.
- Monitor LOS and readmission for high-acuity departments.
- Use Tableau for executive KPI reporting and Streamlit for interactive filtering.
